In [ ]:
# МЕТОД ПОКООРДИНАТНОГО СПУСКА
# f(x1, x2) = x1^2 + 3*x1*x2 + 4*x2^2 + x1 - 17*x2 + 30
# Начальная точка M0 = (2, -3), eps = 0.0001
# На каждом шаге фиксируем одну переменную и минимизируем
# по другой методом дихотомии (половинного деления).

def f(x1, x2):
    return x1**2 + 3*x1*x2 + 4*x2**2 + x1 - 17*x2 + 30

def dichotomy(g, a, b, eps):
    """Метод дихотомии: минимизирует одномерную функцию g на [a, b]."""
    delta = eps / 2
    while (b - a) > eps:
        mid = (a + b) / 2
        x1 = mid - delta
        x2 = mid + delta
        if g(x1) < g(x2):
            b = x2
        else:
            a = x1
    return (a + b) / 2

x = [2.0, -3.0]
eps = 0.0001
R = 30  # полуширина отрезка поиска

print(f'M0 = {x},  f = {f(x[0], x[1]):.6f}')

for k in range(1, 100):
    x_old = x[:]

    # Шаг 1: минимизируем по x1, x2 фиксирован
    g1 = lambda t: f(t, x[1])   # одномерная функция от x1
    x[0] = dichotomy(g1, x[0] - R, x[0] + R, eps)

    # Шаг 2: минимизируем по x2, x1 фиксирован
    g2 = lambda t: f(x[0], t)   # одномерная функция от x2
    x[1] = dichotomy(g2, x[1] - R, x[1] + R, eps)

    print(f'M{k} = ({x[0]:.6f}, {x[1]:.6f}),  f = {f(x[0], x[1]):.6f}')

    dist = ((x[0]-x_old[0])**2 + (x[1]-x_old[1])**2)**0.5
    if dist <= eps:
        print(f'\nСтоп: ||M{k} - M{k-1}|| = {dist:.6f} <= eps')
        break

print(f'\nx* = ({x[0]:.6f}, {x[1]:.6f}),  f(x*) = {f(x[0], x[1]):.6f}')


In [ ]:
# МЕТОД ГРАДИЕНТНОГО СПУСКА
# f(x1, x2) = x1^2 + 3*x1*x2 + 4*x2^2 + x1 - 17*x2 + 30
# Начальная точка M0 = (2, -3), начальный шаг lambda = 0.01, eps = 0.0001
# Если f увеличивается — шаг уменьшается вдвое (дробление шага).

def f(x1, x2):
    return x1**2 + 3*x1*x2 + 4*x2**2 + x1 - 17*x2 + 30

def grad_f(x1, x2):
    return (2*x1 + 3*x2 + 1,
            3*x1 + 8*x2 - 17)

x = [2.0, -3.0]
lam = 0.01
eps = 0.0001

print(f'M0 = {x},  f = {f(x[0], x[1]):.6f}')

f_prev = f(x[0], x[1])
for k in range(1, 100001):
    g1, g2 = grad_f(x[0], x[1])

    # Дробление шага: уменьшаем lambda пока f не убывает
    while True:
        x1_new = x[0] - lam * g1
        x2_new = x[1] - lam * g2
        f_new = f(x1_new, x2_new)
        if f_new <= f_prev:
            break
        lam /= 2

    x = [x1_new, x2_new]
    delta = abs(f_new - f_prev)
    f_prev = f_new

    if k <= 5:
        print(f'M{k} = ({x[0]:.6f}, {x[1]:.6f}),  f = {f_new:.6f}')

    if delta <= eps:
        print(f'\nСтоп на итерации {k}: |f_new - f_prev| = {delta:.6f} <= eps')
        break

print(f'\nx* = ({x[0]:.6f}, {x[1]:.6f}),  f(x*) = {f(x[0], x[1]):.6f}')


In [ ]:
# МЕТОД НАИСКОРЕЙШЕГО СПУСКА
# f(x1, x2) = x1^2 + 3*x1*x2 + 4*x2^2 + x1 - 17*x2 + 30
# Начальная точка M0 = (2, -3), eps = 0.0001
# На каждой итерации:
#   1. Вычисляем градиент grad_f в текущей точке
#   2. Ищем оптимальный шаг h методом дихотомии:
#      минимизируем phi(h) = f(x - h * grad_f) по h
#   3. Делаем шаг: x_new = x - h * grad_f

def f(x1, x2):
    return x1**2 + 3*x1*x2 + 4*x2**2 + x1 - 17*x2 + 30

def grad_f(x1, x2):
    return (2*x1 + 3*x2 + 1,
            3*x1 + 8*x2 - 17)

def dichotomy(g, a, b, eps):
    """Метод дихотомии: минимизирует одномерную функцию g на [a, b]."""
    delta = eps / 2
    while (b - a) > eps:
        mid = (a + b) / 2
        h1 = mid - delta
        h2 = mid + delta
        if g(h1) < g(h2):
            b = h2
        else:
            a = h1
    return (a + b) / 2

x = [2.0, -3.0]
eps = 0.0001

print(f'M0 = {x},  f = {f(x[0], x[1]):.6f}')

for k in range(1, 10000):
    g1, g2 = grad_f(x[0], x[1])
    grad_norm = (g1**2 + g2**2)**0.5

    if grad_norm < eps:
        print(f'Стоп: ||grad f|| = {grad_norm:.6f} < eps')
        break

    # phi(h) = f(x1 - h*g1, x2 - h*g2) — одномерная функция от h
    # минимизируем её дихотомией
    phi = lambda h: f(x[0] - h*g1, x[1] - h*g2)
    h_opt = dichotomy(phi, 0.0, 5.0, eps)

    x = [x[0] - h_opt*g1, x[1] - h_opt*g2]

    if k <= 5:
        print(f'M{k} = ({x[0]:.6f}, {x[1]:.6f}),  f = {f(x[0], x[1]):.6f},  h = {h_opt:.6f}')

print(f'\nx* = ({x[0]:.6f}, {x[1]:.6f}),  f(x*) = {f(x[0], x[1]):.6f}')


In [ ]:
import matplotlib.pyplot as plt
import numpy as np

def f(x1, x2):
    return x1**2 + 3*x1*x2 + 4*x2**2 + x1 - 17*x2 + 30

def grad_f(x):
    return np.array([2*x[0] + 3*x[1] + 1, 3*x[0] + 8*x[1] - 17])

def dichotomy(g, a, b, eps):
    delta = eps / 2
    while (b - a) > eps:
        mid = (a + b) / 2
        if g(mid - delta) < g(mid + delta):
            b = mid + delta
        else:
            a = mid - delta
    return (a + b) / 2

x0 = np.array([2.0, -3.0])
eps = 0.0001

# 1. Покоординатный спуск
def path_coordinate_descent():
    x = x0.copy()
    path = [x.copy()]
    for _ in range(500):
        x_old = x.copy()
        x[0] = dichotomy(lambda t: f(t, x[1]), x[0]-30, x[0]+30, eps)
        path.append(x.copy())
        x[1] = dichotomy(lambda t: f(x[0], t), x[1]-30, x[1]+30, eps)
        path.append(x.copy())
        if np.linalg.norm(x - x_old) < eps:
            break
    return np.array(path)

# 2. Градиентный спуск
def path_gradient_descent():
    x = x0.copy()
    path = [x.copy()]
    lam = 0.01
    f_prev = f(x[0], x[1])
    for _ in range(100000):
        g = grad_f(x)
        cur_lam = lam
        while True:
            x_new = x - cur_lam * g
            f_new = f(x_new[0], x_new[1])
            if f_new <= f_prev:
                break
            cur_lam /= 2
        delta = abs(f_new - f_prev)
        x = x_new
        f_prev = f_new
        path.append(x.copy())
        if delta < eps:
            break
    return np.array(path)

# 3. Наискорейший спуск
def path_steepest_descent():
    x = x0.copy()
    path = [x.copy()]
    for _ in range(500):
        g = grad_f(x)
        if np.linalg.norm(g) < eps:
            break
        phi = lambda h: f(x[0] - h*g[0], x[1] - h*g[1])
        h_opt = dichotomy(phi, 0.0, 5.0, eps)
        x = x - h_opt * g
        path.append(x.copy())
    return np.array(path)

path_cd = path_coordinate_descent()
path_gd = path_gradient_descent()
path_sd = path_steepest_descent()

x1r = np.linspace(-12, 6, 300)
x2r = np.linspace(-4, 10, 300)
X1, X2 = np.meshgrid(x1r, x2r)
Z = X1**2 + 3*X1*X2 + 4*X2**2 + X1 - 17*X2 + 30

configs = [
    ('Покоординатный спуск', path_cd, 'orange'),
    ('Градиентный спуск',    path_gd, 'red'),
    ('Наискорейший спуск',   path_sd, 'blue'),
]

fig, axes = plt.subplots(1, 3, figsize=(21, 7))
x_min = path_sd[-1]

for ax, (name, path, color) in zip(axes, configs):
    levels = np.linspace(Z.min(), Z.min() + 120, 25)
    cp = ax.contour(X1, X2, Z, levels=levels, cmap='viridis', alpha=0.6)
    ax.clabel(cp, inline=True, fontsize=7, fmt='%.0f')
    ax.plot(path[:, 0], path[:, 1], color=color, marker='o', markersize=3, linewidth=1.5)
    ax.plot(x0[0], x0[1], 'ks', markersize=8, label=f'Старт ({x0[0]},{x0[1]})')
    ax.plot(path[-1, 0], path[-1, 1], 'r*', markersize=12,
            label=f'Финиш ({path[-1,0]:.3f}, {path[-1,1]:.3f})')
    ax.set_title(name, fontsize=13)
    ax.set_xlabel('x1')
    ax.set_ylabel('x2')
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print(f'Покоординатный: x*=({path_cd[-1,0]:.6f}, {path_cd[-1,1]:.6f}),  f={f(*path_cd[-1]):.6f}')
print(f'Градиентный:    x*=({path_gd[-1,0]:.6f}, {path_gd[-1,1]:.6f}),  f={f(*path_gd[-1]):.6f}')
print(f'Наискорейший:   x*=({path_sd[-1,0]:.6f}, {path_sd[-1,1]:.6f}),  f={f(*path_sd[-1]):.6f}')
